# Prvi model: klasifikacijsko drevo — A: prikaz (telekom)
**Akademija UI za poslovne aplikacije — Modul 2, Uvod v strojno učenje**

Ta zvezek poganja predavatelj. Vajo naredite v zvezku **`B_klasifikacijsko_drevo`**.

Vprašanje dneva: **ali lahko napovemo, katera stranka bo odšla?**

Najprej se dogovorimo, **kako merimo** napovedi. Nato naučimo prvi model: klasifikacijsko drevo.

### Ponovitev
- **Odločitveno drevo** je zaporedje vprašanj da / ne o značilkah. Na koncu vsake poti je **list** z napovedjo.
- **Učenje:** drevo preizkusi veliko pragov in obdrži tistega, ki najbolje loči stranke, ki ostanejo, od tistih, ki odidejo.
- **Izhodiščni model** vedno napove pogostejši razred. Naš model mora biti boljši od njega.
- **Točnost** (*accuracy*) je delež pravilnih napovedi. Merimo jo na **učni** in na **validacijski** množici.
- **Matrika zamenjav** prešteje štiri izide: napovemo odhod ali ne, stranka odide ali ne.
- **Natančnost** (*precision*): od strank, za katere napovemo odhod, koliko jih res odide?
- **Priklic** (*recall*): od strank, ki res odidejo, koliko jih najdemo?
- **Pretirano prilagajanje:** učna točnost raste, validacijska pa pada. Model si je zapomnil učne podatke.
- **Parametri** so to, kar se model nauči (vprašanja in pragovi). **Hiperparametri** so to, kar nastavimo mi
  (`max_depth`, `min_samples_leaf`).

### Nastavitev okolja
Zaženite spodnji dve celici (**Shift + Enter**). Ničesar ne spreminjajte. Koda prve celice je skrita.
Druga celica pokliče skupno pripravo podatkov iz `utils.modeliranje`. Veljavne pripravljene tabele ponovno uporabi.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("telekom")
shema = preberi_shemo(PODATKI / "shema.json")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plotnine import *
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, SEME
nastavi_slog()

---
## A1. Podatki
Podatke sta pripravila zvezka `A1_Priprava_Telekom` in `A1_Delitev_in_manjkajoce_Telekom`. Skripta
`scripts/pripravi_telekom.py` naredi iste korake in tabele shrani v `data/pripravljeno/delitve/telekom/`. Tu jih samo preberemo.

Model se uči na **učni** množici (`train.csv`). Na **validacijski** (`val.csv`) preverjamo in primerjamo modele.
**Testne** množice (`test.csv`) ne odpremo: ostane zaprta do tekmovanja na koncu klasifikacijskega dela.

<details>
<summary><b>Kako so pripravljeni podatki? (klikni)</b></summary>

Podatke je očistila, razdelila in shranila skripta `scripts/pripravi_telekom.py`. Odločitve z razlogi so v zvezkih
`01_priprava_podatkov/A1_Priprava_Telekom` in `02_preverjanje_modela/A1_Delitev_in_manjkajoce_Telekom`. Na kratko:

1. Ena vrstica = ena stranka (100 000 strank). Cilj je `odhod`.
2. Skrite oznake (`U`, `UNKW`) in nemogoče negativne vrednosti so označene kot manjkajoče.
3. Stolpcem iz prvega modula so dodani novi: skupaj 64 značilk (58 številskih, 6 besedilnih).
4. Delitev 60 / 20 / 20: naključna, z enakim deležem odhodov, seme 42. Vsi imamo iste stranke v istih množicah.
5. Manjkajoče številke so zapolnjene z mediano **učne** množice. Pri izbranih stolpcih stolpec s končnico `_manjka` pove, kje je vrednost manjkala.
6. Besedilni stolpci so zapisani tudi kot stolpci 0/1 (na primer `regija_CHICAGO_AREA`).
   Za model je pripravljenih 96 stolpcev: 58 številskih in 38 stolpcev 0/1.
7. Osamelcev ne odstranimo: drevo vpraša samo, ali je vrednost nad ali pod pragom.
8. Modele primerjamo na validacijski množici. Testna ostane zaprta do tekmovanja na koncu klasifikacijskega dela.

</details>

In [ ]:
ucni = preberi_tabelo(PODATKI / "train.csv", shema)
validacijski = preberi_tabelo(PODATKI / "val.csv", shema)

pd.DataFrame({"strank": [len(ucni), len(validacijski)],
              "delež odhodov": [ucni["odhod"].mean(), validacijski["odhod"].mean()],
              "stolpcev": [ucni.shape[1], validacijski.shape[1]],
              "manjkajočih vrednosti": [ucni.isna().sum().sum(), validacijski.isna().sum().sum()]},
             index=["učna (train.csv)", "validacijska (val.csv)"]).round(3)

V tabeli ni več manjkajočih vrednosti. Primer treh strank: besedilni stolpec `regija` in dva njegova stolpca 0/1,
zapolnjen `dohodek_razred` in oznaka `dohodek_razred_manjka` (1 = vrednost je manjkala in je zapolnjena z mediano).

In [ ]:
primer = pd.concat([ucni[ucni["dohodek_razred_manjka"] == 1].head(2), ucni[ucni["regija"] == "CHICAGO AREA"].head(1)])
primer[["id_stranke", "regija", "regija_CHICAGO_AREA", "regija_DALLAS_AREA", "dohodek_razred", "dohodek_razred_manjka"]]

**Značilke in cilj.** `X` so značilke (kar vemo o stranki), `y` je cilj (`odhod`). `id_stranke` ni značilka.
Za prvi model vzamemo 16 številskih stolpcev, ki jih poznamo iz prvega modula. V tabeli je pripravljenih še več stolpcev;
v zvezku B boste stolpce izbirali sami.

In [ ]:
ZNACILKE = ["mesecev_narocnine", "povp_mesecni_prihodek", "povp_mesecna_narocnina", "povp_minute",
            "povp_minute_6m", "sprememba_minut", "povp_minute_prekoracitev", "povp_gostovanje",
            "povp_klici_podpora", "povp_prekinjeni_klici", "stevilo_narocnin", "stevilo_telefonov",
            "dni_z_napravo", "cena_naprave", "dohodek_razred", "stevilo_avtomobilov"]

X_ucni, y_ucni = ucni[ZNACILKE], ucni["odhod"]
X_val, y_val = validacijski[ZNACILKE], validacijski["odhod"]

print("X_ucni:", X_ucni.shape, "  X_val:", X_val.shape)
print("Značilk:", len(ZNACILKE), "  manjkajočih vrednosti v X_ucni:", int(X_ucni.isna().sum().sum()))

## A2. Kako merimo napovedi?
Preden naučimo model, se dogovorimo, **kako ga bomo ocenili**. Mere preizkusimo na dveh preprostih "modelih", ki se ne učita:

1. **izhodišče:** vedno napovemo pogostejši razred, ne glede na značilke;
2. **ročno pravilo iz 1. modula:** stranka odide, če ima napravo več kot 300 dni.

Najprej izhodišče. Koliko napovedi bi zadelo?

In [ ]:
pogostejsi = y_ucni.mode()[0]                   # pogostejši razred v učnih podatkih
izhodisce = (y_val == pogostejsi).mean()        # delež pravilnih napovedi na validacijski množici

print("Pogostejši razred:", pogostejsi, "(0 = ostane, 1 = odide)")
print(f"Točnost izhodišča: {izhodisce:.3f}")

napoved_izhodisce = np.full(len(y_val), pogostejsi)            # vsem strankam napovemo isto

Polovica strank je odšla, zato izhodišče zadene približno polovico. Toda izhodišče napove, da **vse stranke ostanejo**.
Ne najde niti ene stranke, ki odide. Točnost tega ne pokaže. Zato potrebujemo še druge mere.

Zdaj ročno pravilo iz 1. modula: `1` = napovemo odhod, `0` = napovemo, da stranka ostane.

In [ ]:
napoved_pravilo = (X_val["dni_z_napravo"] > 300).astype(int)      # ročno pravilo iz 1. modula

# matrika zamenjav: vrstice = kaj se je zgodilo, stolpci = kaj smo napovedali
pd.crosstab(y_val.map({1: "odide", 0: "ostane"}).rename("dejansko"),
            napoved_pravilo.map({1: "odide", 0: "ostane"}).rename("napoved"))

### Matrika zamenjav (*confusion matrix*)
Vsaka stranka pade v eno od štirih polj. Napovedi "odide" rečemo **pozitivna**, ker iščemo prav te stranke.
Za polja uporabljamo angleške kratice, ki jih boste srečali povsod:

| | napoved: odide | napoved: ostane |
|---|---|---|
| **dejansko: odide** | **TP** (*true positive*, resnično pozitivni): pravilno najdena | **FN** (*false negative*, lažno negativni): spregledana |
| **dejansko: ostane** | **FP** (*false positive*, lažno pozitivni): lažni alarm | **TN** (*true negative*, resnično negativni): pravilno pri miru |

Iz štirih števil izračunamo tri mere:
- **točnost** (*accuracy*) = (TP + TN) / vse stranke: delež pravilnih napovedi;
- **natančnost** (*precision*) = TP / (TP + FP): od strank, za katere napovemo odhod, koliko jih res odide;
- **priklic** (*recall*) = TP / (TP + FN): od strank, ki res odidejo, koliko jih najdemo.

Funkcija `mere` prešteje štiri polja in izračuna vse tri mere. `graf_matrike` matriko nariše. Kopiji sta v `utils/orodja.py`.

In [ ]:
def mere(dejansko, napoved):
    """Matrika zamenjav in tri mere. 1 = odide (to napoved imenujemo pozitivna), 0 = ostane."""
    dejansko, napoved = np.asarray(dejansko), np.asarray(napoved)
    tp = int(((napoved == 1) & (dejansko == 1)).sum())      # TP, true positive: napovemo odhod, stranka odide
    fp = int(((napoved == 1) & (dejansko == 0)).sum())      # FP, false positive: napovemo odhod, stranka ostane
    fn = int(((napoved == 0) & (dejansko == 1)).sum())      # FN, false negative: napovemo, da ostane, stranka odide
    tn = int(((napoved == 0) & (dejansko == 0)).sum())      # TN, true negative: napovemo, da ostane, in ostane
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "točnost": (tp + tn) / len(dejansko),                           # delež pravilnih napovedi
            "natančnost": tp / (tp + fp) if tp + fp > 0 else np.nan,       # brez napovedanih odhodov ni določena
            "priklic": tp / (tp + fn)}                                       # delež najdenih odhodov


def graf_matrike(dejansko, napoved, naslov=""):
    """Matrika zamenjav kot graf. Vrstice: kaj se je zgodilo. Stolpci: kaj smo napovedali."""
    m = mere(dejansko, napoved)
    polja = pd.DataFrame({"napoved": ["odide", "ostane", "odide", "ostane"],
                          "dejansko": ["odide", "odide", "ostane", "ostane"],
                          "oznaka": ["TP", "FN", "FP", "TN"]})
    polja["strank"] = [m[o] for o in polja["oznaka"]]
    polja["izid"] = np.where(polja["napoved"] == polja["dejansko"], "pravilno", "napaka")
    polja["besedilo"] = [f"{o}\n{s:,}".replace(",", " ") for o, s in zip(polja["oznaka"], polja["strank"])]
    return (ggplot(polja, aes(x="napoved", y="dejansko", fill="izid"))
            + geom_tile(color="white", size=2)
            + geom_text(aes(label="besedilo"), size=13)
            + scale_fill_manual(values={"pravilno": "#cfe8d9", "napaka": "#f6d5c7"})
            + scale_x_discrete(limits=["odide", "ostane"])
            + scale_y_discrete(limits=["ostane", "odide"])            # "odide" je zgoraj
            + coord_equal()
            + labs(title=naslov, x="napoved (predicted)", y="dejansko (actual)", fill="")
            + theme(panel_grid=element_blank(), axis_text=element_text(size=11), figure_size=(4.6, 3.6)))

graf_matrike(y_val, napoved_pravilo, "Ročno pravilo: dni z napravo > 300")

Obe napovedi primerjamo v eni tabeli. Dodamo še dve:
- drugo skrajno izhodišče: **vsem** strankam napovemo, da **odidejo**;
- drugo ročno pravilo: stranka odide, če ima na mesec **največ 20 minut** klicev.

In [ ]:
napoved_vsi_odidejo = np.ones(len(y_val), dtype=int)               # drugo skrajno izhodišče: vsem napovemo odhod
napoved_minute = (X_val["povp_minute"] <= 20).astype(int)        # drugo pravilo: skoraj ne kliče

pd.DataFrame([mere(y_val, napoved_izhodisce), mere(y_val, napoved_vsi_odidejo), mere(y_val, napoved_pravilo), mere(y_val, napoved_minute)],
             index=["izhodišče: vsi ostanejo", "izhodišče: vsi odidejo", "pravilo: dni z napravo > 300", "pravilo: minute <= 20"]).round(3)

- **Vsi ostanejo:** točnost 0,504, priklic **0**: ne najde nobene stranke, ki odide. Natančnosti ne moremo izračunati
  (`NaN`), ker odhoda ne napove nikomur.
- **Vsi odidejo:** priklic **1,000**: najde vse stranke, ki odidejo. Natančnost pa je 0,496, toliko kot delež odhodov,
  in točnost 0,496. Popoln priklic je lahko dobiti in sam nič ne pomeni: alarm sproži za vsakogar.
- **Pravilo:** točnost 0,580. Od strank, za katere napove odhod, jih odide 56 % (natančnost 0,562).
  Najde 69 % vseh strank, ki odidejo (priklic 0,692). Za to pa sproži 5 338 lažnih alarmov.

- **Pravilo z minutami:** točnost samo 0,518 in priklic 0,071: najde 7 % strank, ki odidejo.
  Ko pa napove odhod, ima prav v 62 % primerov (natančnost 0,624), pogosteje kot pravilo z napravo.

Ena številka ne pove vsega. Katera mera je pomembna, je odvisno od vprašanja: ali želimo najti **čim več** strank, ki odidejo
(priklic), ali želimo, da so naši alarmi **čim bolj zanesljivi** (natančnost)?

> **Povezava s 1. modulom:** videli smo, da je `odhod` v teh podatkih namenoma uravnotežen (50 : 50).
> Zato je točnost tu uporabna za prvi pregled. Če bi odšlo le 5 % strank, bi imelo izhodišče točnost 0,95,
> priklic pa bi bil še vedno 0.

Preverimo: iste številke vrne `sklearn.metrics`. Matrika ima tu vrstni red `[odide, ostane]`, kot v tabeli zgoraj.

In [ ]:
print(confusion_matrix(y_val, napoved_pravilo, labels=[1, 0]))       # [[TP, FN], [FP, TN]]
print(f"točnost {accuracy_score(y_val, napoved_pravilo):.3f}   natančnost {precision_score(y_val, napoved_pravilo):.3f}   "
      f"priklic {recall_score(y_val, napoved_pravilo):.3f}")
print("izhodišče: priklic", recall_score(y_val, napoved_izhodisce),
      "  natančnost", precision_score(y_val, napoved_izhodisce, zero_division=np.nan))
print(f"vsi odidejo: priklic {recall_score(y_val, napoved_vsi_odidejo):.3f}   natančnost {precision_score(y_val, napoved_vsi_odidejo):.3f}")

## A3. Drevo z enim vprašanjem
Model v scikit-learn vedno uporabimo v dveh korakih: **ustvarimo** ga z nastavitvami in ga **naučimo** s `fit(X, y)`.
`max_depth=1` pomeni: drevo sme postaviti samo eno vprašanje.

In [ ]:
drevo_1 = DecisionTreeClassifier(max_depth=1, random_state=SEME)
drevo_1.fit(X_ucni, y_ucni)

plt.figure(figsize=(7, 3.2))
plot_tree(drevo_1, feature_names=ZNACILKE, class_names=["ostane", "odide"], impurity=False, fontsize=10)
plt.show()

Kako beremo vozlišče:
- prva vrstica je **vprašanje** (levo = da, desno = ne),
- `samples` je število strank v vozlišču,
- `value` = [koliko jih ostane, koliko jih odide],
- `class` je napoved (pogostejši razred v vozlišču).

Vprašanje v drevesu ima tudi strokovno ime: **razcep** (angl. *split*). Vozlišče stranke razcepi na dve skupini.

Drevo je med 16 značilkami in vsemi možnimi pragovi izbralo **`dni_z_napravo <= 304,5`**.

> **Povezava s 1. modulom:** to je skok, ki smo ga takrat našli ročno, na grafu. Model ga je našel sam.

Narišimo isti graf in vanj prag drevesa:

In [ ]:
prag = drevo_1.tree_.threshold[0]                                     # prag v korenu drevesa

razredi = pd.cut(ucni["dni_z_napravo"], bins=range(0, 801, 50))       # razredi po 50 dni
delez = ucni.groupby(razredi, observed=True)["odhod"].mean().mul(100).reset_index(name="odstotek")
delez["dni"] = [razred.mid for razred in delez["dni_z_napravo"]]

(ggplot(delez, aes(x="dni", y="odstotek"))
 + geom_col(fill=MODRA)
 + geom_vline(xintercept=prag, color=ORANZNA, linetype="dashed", size=1.3)
 + labs(title="Delež odhodov glede na dni z napravo",
        subtitle=f"oranžna črta: prag, ki ga je našlo drevo ({prag:.1f} dni)",
        x="dni z napravo", y="% strank, ki so odšle"))

**Kako drevo izmeri razcep?** Za vsako skupino strank izračuna **Ginijevo nečistočo**:

`Gini = 1 − p² − (1 − p)²`, kjer je `p` delež strank v skupini, ki odidejo.

Če so v skupini vsi enaki, je Gini 0. Če jih pol ostane in pol odide, je Gini 0,5 (največ).
Dober razcep naredi dve skupini, ki sta bolj čisti od prvotne. Drevo za vsak možen razcep izračuna uteženo povprečje
Ginija leve in desne skupine in obdrži razcep z **največjim zmanjšanjem**.

Izračunajmo sami: najprej razcep, ki ga je drevo izbralo, nato za primerjavo vprašanje `povp_minute <= 358`
(ali stranka kliče manj od mediane).

In [ ]:
def gini(odhod):
    """Ginijeva nečistoča skupine: 0 = vsi enaki, 0,5 = pol ostane in pol odide."""
    p = odhod.mean()                                        # delež strank, ki odidejo
    return 1 - p**2 - (1 - p)**2

def izmeri_razcep(stolpec, prag):
    """Izpiše Gini pred razcepom, v levi in desni skupini ter zmanjšanje."""
    vsi = ucni["odhod"]
    levo = ucni.loc[ucni[stolpec] <= prag, "odhod"]         # stranke z odgovorom da
    desno = ucni.loc[ucni[stolpec] > prag, "odhod"]         # stranke z odgovorom ne
    potem = (len(levo) * gini(levo) + len(desno) * gini(desno)) / len(vsi)      # uteženo povprečje
    print(f"{stolpec} <= {prag}")
    for ime, skupina in [("prej", vsi), ("levo (da)", levo), ("desno (ne)", desno)]:
        print(f"   {ime:11s}{len(skupina):6d} strank, odide {skupina.mean():.1%}, Gini {gini(skupina):.4f}")
    print(f"   potem (uteženo povprečje leve in desne): {potem:.4f}    zmanjšanje: {gini(vsi) - potem:.4f}\n")

izmeri_razcep("dni_z_napravo", 304.5)       # razcep, ki ga je izbralo drevo
izmeri_razcep("povp_minute", 358)           # drugo vprašanje, za primerjavo
print("Gini v korenu, levo in desno, kot jih je izračunalo drevo:", drevo_1.tree_.impurity.round(4))

Izbrani razcep zmanjša Gini za 0,0140, vprašanje o minutah samo za 0,0014: desetkrat manj.
Naši izračuni se ujemajo s številkami, ki jih hrani drevo (`drevo_1.tree_.impurity`).

In [ ]:
print(f"Izhodišče:            {izhodisce:.3f}")
print(f"Drevo, 1 vprašanje:   učna {drevo_1.score(X_ucni, y_ucni):.3f}   validacijska {drevo_1.score(X_val, y_val):.3f}")

Eno samo vprašanje dvigne točnost z 0,50 na 0,58. `score` vrne točnost: delež pravilnih napovedi.

## A4. Drevo globine 3: beremo pravila
Za lepši izris uporabimo funkcijo `narisi_drevo` iz `utils/orodja.py`. Znotraj kliče isti `plot_tree`,
oznake pa prevede: modro = večina ostane, oranžno = večina odide.

In [ ]:
from utils.orodja import narisi_drevo

drevo_3 = DecisionTreeClassifier(max_depth=3, random_state=SEME).fit(X_ucni, y_ucni)

narisi_drevo(drevo_3, ZNACILKE)
plt.show()

> **Povezava s 1. modulom:** drugo vprašanje na levi je `mesecev_narocnine <= 10,5`. To je drugi skok, ki smo ga našli
> ročno ("pri 11 mesecih"). Korelacija te značilke z odhodom je skoraj 0 — drevo vzorec vseeno najde,
> ker ne išče premice, ampak prag.

Korelacija in ista pravila kot besedilo:

In [ ]:
print("Korelacija mesecev_narocnine z odhodom: r =", round(ucni["mesecev_narocnine"].corr(ucni["odhod"]), 2))
print()
print(export_text(drevo_3, feature_names=ZNACILKE, decimals=1))

**Pot ene stranke.** Vzamemo eno stranko iz validacijske množice in ji sledimo od korena do lista.
Model se lahko tudi zmoti: v listu, kjer odide 54 % strank, jih 46 % ostane.

In [ ]:
stranka = X_val.iloc[[0]]                              # prva stranka iz validacijske množice

for vozlisce in drevo_3.decision_path(stranka).indices[:-1]:      # vozlišča na poti, brez lista
    znacilka = ZNACILKE[drevo_3.tree_.feature[vozlisce]]
    prag = drevo_3.tree_.threshold[vozlisce]
    vrednost = stranka[znacilka].iloc[0]
    print(f"{znacilka} = {vrednost:g}   ali je <= {prag:.1f}?   {'da' if vrednost <= prag else 'ne'}")

print("Napoved:", drevo_3.predict(stranka)[0], "  dejansko:", y_val.loc[stranka.index[0]], "  (0 = ostane, 1 = odide)")

## A5. Napovedi
`predict` vrne razred, `predict_proba` pa verjetnost. Verjetnost odhoda je **delež strank v listu, ki so odšle**.
Drevo globine 3 ima 8 listov, zato pozna samo 8 različnih verjetnosti.

In [ ]:
pet_strank = X_val.sample(5, random_state=6)           # pet naključnih strank iz validacijske množice

pd.DataFrame({"napoved": drevo_3.predict(pet_strank),
              "verjetnost odhoda": drevo_3.predict_proba(pet_strank)[:, 1].round(2),
              "dejansko": y_val.loc[pet_strank.index]})

## A6. Kako dober je model?
Točnost izračunamo ročno (delež pravilnih napovedi) in s `score`. Vedno jo primerjamo z izhodiščem.

In [ ]:
napovedi = drevo_3.predict(X_val)
print("Ročno:", (napovedi == y_val).mean(), "  s score:", drevo_3.score(X_val, y_val))

pd.DataFrame({"učna": [np.nan, drevo_1.score(X_ucni, y_ucni), drevo_3.score(X_ucni, y_ucni)],
              "validacijska": [izhodisce, drevo_1.score(X_val, y_val), drevo_3.score(X_val, y_val)]},
             index=["izhodišče", "drevo, globina 1", "drevo, globina 3"]).round(3)

Zdaj še vse tri mere iz A2, za obe drevesi, ob izhodišču in ročnem pravilu:

In [ ]:
pd.DataFrame([mere(y_val, napoved_izhodisce), mere(y_val, napoved_pravilo),
              mere(y_val, drevo_1.predict(X_val)), mere(y_val, napovedi)],
             index=["izhodišče: vsi ostanejo", "pravilo: dni z napravo > 300", "drevo, globina 1", "drevo, globina 3"]).round(3)

In [ ]:
graf_matrike(y_val, napovedi, "Drevo globine 3 na validacijski množici")

- Drevo z enim vprašanjem je skoraj isto kot ročno pravilo: prag je 304,5 namesto 300 dni.
- Drevo globine 3 ima malo večjo točnost (0,589) in **večji priklic** (0,789): najde 7 823 od 9 913 strank, ki odidejo.
  Natančnost pa je skoraj enaka (0,560): skoraj vsak drugi alarm je lažen.
- Model je boljši od izhodišča in od pravila, vendar ne pri vseh merah enako. Kako napovedi uporabimo pri
  omejenem številu klicev, pogledamo v sklopu **Od napovedi do poslovne odločitve**.

**Osamelci.** Pri pripravi podatkov velikih vrednosti nismo odstranili (zvezek `A1_Priprava_Telekom`). Zdaj imamo drevo in lahko to odločitev preverimo. Vrednosti obrežemo na 1. in 99. percentil
učnih podatkov in drevo naučimo še enkrat.

> **Povezava s 1. modulom:** osamelce smo iskali s škatlastim diagramom. Povprečje so močno premaknili, mediane skoraj nič.
> Drevo se obnaša kot mediana: pomemben je vrstni red vrednosti, ne njihova velikost.

In [ ]:
spodaj, zgoraj = X_ucni.quantile(0.01), X_ucni.quantile(0.99)       # meje izračunamo samo na učnih podatkih
X_ucni_obrezano = X_ucni.clip(spodaj, zgoraj, axis=1)
X_val_obrezano = X_val.clip(spodaj, zgoraj, axis=1)
drevo_obrezano = DecisionTreeClassifier(max_depth=3, random_state=SEME).fit(X_ucni_obrezano, y_ucni)

print(f"Minute na mesec: mediana {X_ucni['povp_minute'].median():.0f}, največ {X_ucni['povp_minute'].max():.0f}, po obrezovanju največ {X_ucni_obrezano['povp_minute'].max():.0f}")
print(f"Drevo globine 3, validacijska točnost: z osamelci {drevo_3.score(X_val, y_val):.3f}, z obrezanimi vrednostmi {drevo_obrezano.score(X_val_obrezano, y_val):.3f}")

Na tej validacijski množici je točnost po obrezovanju enaka. V tem poskusu obrezovanje ni pomagalo,
zato pri nadaljnjih drevesih obdržimo izvorne vrednosti. Iz enega poskusa ne sklepamo, da osamelci drevesom nikoli ne škodijo.

## A7. Drevo kot zemljevid
Drevo prostor značilk razreže na pravokotnike. Pri dveh značilkah lahko to narišemo.

**Pozor:** to je **ločeno drevo**, naučeno samo na dveh značilkah (`dni_z_napravo` in `mesecev_narocnine`).
Drevesa s 16 značilkami na ravnini ne moremo narisati.

In [ ]:
def graf_obmocij(drevo, podatki, x, y, koraki=160):
    """Območja napovedi za drevo, ki je naučeno SAMO na dveh značilkah (x in y)."""
    meje_x = podatki[x].quantile([0.005, 0.995])
    meje_y = podatki[y].quantile([0.005, 0.995])
    mreza = pd.DataFrame([(a, b) for a in np.linspace(*meje_x, koraki) for b in np.linspace(*meje_y, koraki)],
                         columns=[x, y])
    mreza["verjetnost"] = drevo.predict_proba(mreza)[:, 1]
    return (ggplot(mreza, aes(x=x, y=y, fill="verjetnost"))
            + geom_tile()
            + scale_fill_gradient2(low=MODRA, mid="#f0efec", high=ORANZNA, midpoint=0.5, limits=(0, 1))
            + labs(fill="verjetnost\nodhoda")
            + theme(panel_grid=element_blank()))

DVE = ["dni_z_napravo", "mesecev_narocnine"]
drevo_dve = DecisionTreeClassifier(max_depth=3, random_state=SEME).fit(X_ucni[DVE], y_ucni)
print(f"Drevo na dveh značilkah: validacijska točnost {drevo_dve.score(X_val[DVE], y_val):.3f}")

graf_obmocij(drevo_dve, X_ucni, "dni_z_napravo", "mesecev_narocnine") + labs(title="Drevo globine 3 na dveh značilkah")

Vsak pravokotnik je en list. Oranžno: večina strank odide. Modro: večina ostane.

## A8. Drevo brez omejitve
Kaj se zgodi, če drevesu dovolimo poljubno veliko vprašanj?

In [ ]:
drevo_brez = DecisionTreeClassifier(random_state=SEME).fit(X_ucni, y_ucni)       # brez max_depth

print("Listov:", drevo_brez.get_n_leaves(), "  globina:", drevo_brez.get_depth())
print(f"Učna točnost:         {drevo_brez.score(X_ucni, y_ucni):.3f}")
print(f"Validacijska točnost: {drevo_brez.score(X_val, y_val):.3f}   (izhodišče {izhodisce:.3f})")

Na učnih podatkih je drevo popolno, na novih strankah pa komaj boljše od izhodišča.
Drevo si je učne stranke **zapomnilo**, namesto da bi se naučilo splošna pravila. To je **pretirano prilagajanje**.

Isto na zemljevidu (spet ločeno drevo na dveh značilkah, tokrat brez omejitve):

In [ ]:
drevo_dve_brez = DecisionTreeClassifier(random_state=SEME).fit(X_ucni[DVE], y_ucni)
print("Listov:", drevo_dve_brez.get_n_leaves())

graf_obmocij(drevo_dve_brez, X_ucni, "dni_z_napravo", "mesecev_narocnine") + labs(title="Drevo brez omejitve na dveh značilkah")

## A9. Katera globina je prava?
Naučimo drevo za vsako globino od 1 do 20 in obakrat izmerimo točnost. Zanko zapakiramo v funkcijo
(kopija je v `utils/orodja.py`).

In [ ]:
def oceni_globine(X_ucni, y_ucni, X_val, y_val, globine=range(1, 21)):
    """Za vsako globino nauči drevo in vrne točnost na učni in validacijski množici."""
    vrstice = []
    for globina in globine:
        drevo = DecisionTreeClassifier(max_depth=globina, random_state=SEME).fit(X_ucni, y_ucni)
        vrstice.append({"globina": globina, "listov": drevo.get_n_leaves(),
                        "učna": drevo.score(X_ucni, y_ucni), "validacijska": drevo.score(X_val, y_val)})
    return pd.DataFrame(vrstice)

rezultati = oceni_globine(X_ucni, y_ucni, X_val, y_val)
rezultati.round(3)

In [ ]:
dolga = rezultati.melt(id_vars="globina", value_vars=["učna", "validacijska"], var_name="množica", value_name="točnost")

(ggplot(dolga, aes(x="globina", y="točnost", color="množica"))
 + geom_line(size=1.2)
 + geom_point(size=2)
 + scale_color_manual(values={"učna": MODRA, "validacijska": ORANZNA})
 + labs(title="Točnost glede na globino drevesa", x="globina (max_depth)", y="točnost", color=""))

Učna točnost med preizkušenimi globinami raste. Na tej delitvi ima **validacijska** točnost vrh okoli globine 8,
nato pada. Globino izberemo po validacijski krivulji; testna množica ostane za končno oceno.

## A10. Parametri in hiperparametri
- **Parametri:** vprašanja in pragovi (`dni_z_napravo <= 304,5`). Naučil se jih je model s `fit`.
- **Hiperparametri:** `max_depth`, `min_samples_leaf` (najmanj strank v listu). Nastavimo jih mi, pred učenjem.

---
Naprej: zvezek **`B_klasifikacijsko_drevo`** — drevesa spreminjate sami.